# Script 08: Model

## Preamble

In [ ]:
# Importing packages

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.optimize import minimize

In [ ]:
# Plot functions

# Default formatting for plots
FIGSIZE = (9, 3)
def plot_format():
    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(FIGSIZE[0], FIGSIZE[1] * 1.8),
        gridspec_kw={"height_ratios": [1.5, 1]}, sharex=True
    )
    ax1.set_ylabel("Optimal Party Loyalty\n(percentage points)")
    ax2.set_xlabel("Term")
    ax2.set_ylabel("Relative to Immediate Vesting\n(percentage points)")
    ax2.axhline(y=0, color="black", linestyle="--", linewidth=1, zorder=1)
    return fig, ax1, ax2


# Plot the difference between the tau-vesting path and the immediate-vesting (tau=1) counterfactual
def plot_counterfactual(ax2, panel, panel_cf, color="black"):
    cf = panel_cf.set_index("term")["loyalty"].reindex(panel["term"]).values

    ax2.scatter(panel["term"], 100*(panel["loyalty"].values - cf), color=color, zorder=2)

## Numerical Model Implementation
I normalize $s,b,s_O,C_{100}$ by $s$. This is possible because the optimal loyalty path is invariant to scaling $s_t, b_t, P_{V,t}, P_{I,t}, V_{O,t+1}, \delta_t$ by a positive constant, because I assume $s,b,\delta$ are fixed, and because $P_{V,t}, P_{I,t}, V_{O,t+1}, \delta$ are linear functions of $s,s_O,C_{100}$.

In [ ]:
# External parameters

s = 1 # normalization
s_O = 1.89 # ratio of external salary to Congressional salary

r_A = 0.017 # annual pension annuity accrual rate
tau = 3 # term in which pension vests
col = 62 # age when pension collection can start

In [ ]:
# Default legislator parameters

alpha = 0.9 # share of votes legislator wants to side with their party on
beta = 0.972 # term (2 year) discount factor

age = 48 # age in first term
ret = 64 # planned retirement age, must be >= col

b = 1.207 # ratio of other benefits of office to Congressional salary
C_100 = 0.1 # cognitive cost of fully supporting the party for 1 term, divided by Congressional salary

In [ ]:
# Bellman backward induction solver

def bellman_solve(P_I, delta, beta, alpha, T, b, tau, F, s):
    l_opt = [None] * T
    V = [None] * T
    V_O = [None] * T
    D = [0] * T

    # Last term: no continuation value, so optimal loyalty = alpha
    t = T - 1
    pension = ((t+1==tau)*tau*P_I + (t+1>tau)*P_I) * beta
    l_opt[t] = alpha
    V[t] = s + b + pension

    # Backward induction for remaining terms
    for t in range(T-2, -1, -1):
        V_O[t+1] = s_O*np.sum([beta**(i) for i in range(0, T-t-1)])
        pension = ((t+1==tau)*tau*P_I + (t+1>tau)*P_I) * beta**(T-t)

        def term_value(l_t, pension=pension, V_next=V[t+1], V_O_next=V_O[t+1]):
            C_t = F(l_t)*V_next + (1-F(l_t))*V_O_next
            return s + b + pension - delta*(l_t - alpha)**2 + beta*max(C_t, V_O_next)

        result = minimize(lambda l: -term_value(l), [alpha], method="Powell", bounds=[(0,1)])
        l_opt[t] = result.x[0]
        V[t] = term_value(l_opt[t])
        D[t+1] = V[t+1] - V_O[t+1]

    # manually calculate outside option and difference in values at term 1
    V_O[0] = s_O*np.sum([beta**(i) for i in range(0, T)])
    D[0] = V[0] - V_O[0]

    return l_opt, V, V_O, D

In [ ]:
# Solution and results function

def solve_model(F, alpha=alpha, C_100=C_100, age=age, beta=beta, ret=ret, b=b, tau=tau, col=col, s=s, r_A=r_A):

    # calculating input values
    col = max(col,ret) # if planned retirement is after collection age, use planned retirement
    T = int(np.ceil((col - age)/2)) # terms until retirement & pension collection
    retire_age = age + 2*T # actual age when they leave office, accounting for finishing a partial term
    P_I = r_A*s*np.sum([beta**(i/2) for i in range(0,80-retire_age)]) # calculate term pension increase PDV with 80 year life expectancy
    delta = C_100/((1-alpha)**2) # calculate delta based on C and alpha

    # solve via Bellman backward induction
    l_opt, V, V_O, D = bellman_solve(P_I, delta, beta, alpha, T, b, tau, F, s)

    # handling decision of whether to run for re-election and creating panel dataframe
    df = pd.DataFrame({"term": range(1,T+1), "loyalty": l_opt, "term_value": V, "outside_value": V_O, "continuation_diff": D})
    for t in range(1,T):
        if df.loc[t, "continuation_diff"] <= 0:
            df.loc[t:, "loyalty"] = np.nan # if outside option is preferred at time t, future decisions are irrelevant
            break

    return df, T

## Bliss Point $F'(\alpha)<0$

In [ ]:
# Default parameters with bliss point of re-election likelihood function

F_bliss = lambda x: 1-(x-0.8)**2 # Re-election likelihood maximized at 80% loyalty

panel_dec, T = solve_model(F=F_bliss)

# Counterfactual: same model, but pension vests immediately (tau = 1)
panel_dec_cf, _ = solve_model(F=F_bliss, tau=1)

fig_bliss_F, ax1, ax2 = plot_format()

# Plot
ax1.scatter(panel_dec["term"], 100*panel_dec["loyalty"], color = "black", zorder = 2)

ax1.axvline(x=tau, ymin=0, ymax=1, color = "navy", linestyle = "--", zorder = 1)

ax1.set_xlim(left = 0.75)
ax1.set_xticks(range(1,T+1))
ax1.set_yticks(range(86,92,1))

plot_counterfactual(ax2, panel_dec, panel_dec_cf)

ax2.axvline(x=tau, ymin=0, ymax=1, color = "navy", linestyle = "--", zorder = 1)
ax2.set_yticks(np.arange(-0.3,0.1,0.1))
ax2.set_ylim(top = 0.1)

fig_dec = plt.gcf()
plt.show()
print("\nAverage value difference:", np.mean(panel_dec["continuation_diff"]))

## Strictly Increasing $F'(\alpha)>0$

In [ ]:
# Default parameters with strictly increasing re-election likelihood perception function

F_inc = lambda x: 1-(x-1)**2 # Re-election likelihood maximized at 100% loyalty

panel_inc, T = solve_model(F=F_inc)

# Counterfactual: same model, but pension vests immediately (tau = 1)
panel_inc_cf, _ = solve_model(F=F_inc,tau=1)

fig_default, ax1, ax2 = plot_format()

# Plot
ax1.scatter(panel_inc["term"], 100*panel_inc["loyalty"], color = "black", zorder = 2)

ax1.axvline(x=tau, ymin=0, ymax=1, color = "navy", linestyle = "--", zorder = 1)

ax1.set_xlim(left = 0.75)
ax1.set_xticks(range(1,T+1))
ax1.set_yticks(range(89,95,1))

plot_counterfactual(ax2, panel_inc, panel_inc_cf)

ax2.axvline(x=tau, ymin=0, ymax=1, color = "navy", linestyle = "--", zorder = 1)
ax2.set_yticks(np.arange(0,0.4,0.1))
ax2.set_ylim(bottom = -0.1)

fig_inc = plt.gcf()
plt.show()

print("\nAverage value difference:", np.mean(panel_inc["continuation_diff"]))

In [ ]:
# Combined plot: counterfactual-difference panel from both cases, sharing x-axis and y-axis title

fig_combined_cf, (ax_dec, ax_inc) = plt.subplots(
    2, 1, figsize=(FIGSIZE[0], FIGSIZE[1] * 1.8), sharex=False
)

for ax in (ax_dec, ax_inc):
    ax.axhline(y=0, color="black", linestyle="--", linewidth=1, zorder=1)
    ax.axvline(x=tau, ymin=0, ymax=1, color="navy", linestyle="--", zorder=1)

plot_counterfactual(ax_dec, panel_dec, panel_dec_cf)
ax_dec.set_yticks(np.arange(-0.3,0.1,0.1))
ax_dec.set_ylim(top = 0.1)

ax_dec.set_xlabel("Term")
ax_dec.set_xlim(left = 0.75)
ax_dec.set_xticks(range(1,T+1))
ax_dec.set_title(r"$F'(\alpha)<0$")

plot_counterfactual(ax_inc, panel_inc, panel_inc_cf)
ax_inc.set_yticks(np.arange(0,0.4,0.1))
ax_inc.set_ylim(bottom = -0.1)

ax_inc.set_xlabel("Term")
ax_inc.set_xlim(left = 0.75)
ax_inc.set_xticks(range(1,T+1))
ax_inc.set_title(r"$F'(\alpha)>0$")

fig_combined_cf.supylabel("Party Loyalty Relative to Immediate Vesting\n(percentage points)", ma='center')
fig_combined_cf.tight_layout(rect=[0.03, 0, 1, 1])

plt.show()

In [ ]:
# Save plots

fig_inc.savefig("plots/increase_F.pdf", bbox_inches='tight')
fig_dec.savefig("plots/decrease_F.pdf", bbox_inches='tight')
fig_combined_cf.savefig("plots/combined_cf.pdf", bbox_inches='tight')